In [4]:
import random
import numpy as np
import sys
from scipy.sparse import csr_matrix, hstack, vstack
import time

# --- インポートの一本化と表示設定 ---
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
L=12
l_h = L // 2
J=3
P=768
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [5]:
class ImprovedHighEntropyAPM:
    def __init__(self, P=768, L_half=6, J=3):
        self.P = P
        self.L_half = L_half
        self.J = J
        self.mid = P // 2
        # A領域(0~mid-1)とB領域(mid~P-1)の独立した基底サイクル
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))

    def _generate_random_cycle(self, r):
        indices = list(r)
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def compose(self, p1, p2):
        return tuple(p1[p2[i]] for i in range(self.P))

    def get_power(self, base_p, k):
        res = tuple(range(self.P))
        curr = base_p
        k %= (self.mid if base_p == self.rho_A else (self.P - self.mid))
        while k > 0:
            if k % 2 == 1: res = self.compose(res, curr)
            curr = self.compose(curr, curr)
            k //= 2
        return res

    def solve(self):
        print(f"--- 改善版 HighEntropy 探索開始 (P={self.P}, Girth向上モデル) ---")
        start_time = time.time()
        trial = 0
        while True:
            trial += 1
            # 1. 独立した領域からベース指数を生成
            f_indices = [random.randint(1, self.mid-1) for _ in range(self.L_half)]
            g_indices = [random.randint(1, self.mid-1) for _ in range(self.L_half)]
            
            F = [self.get_power(self.rho_A, k) for k in f_indices]
            G = [self.get_power(self.rho_B, k) for k in g_indices]

            # 2. ターゲット・非可換ノイズの注入 (f0-g3, f1-g2)
            # f0 に A領域の強い攪乱を入れる
            F[0] = self.compose(F[0], self._generate_random_cycle(range(0, self.mid)))
            # g3 に A領域の成分を少量混ぜて f0 と衝突させる
            G[3] = self.compose(G[3], self.get_power(self.rho_A, random.randint(1, 5)))

            # f1 に B領域の成分を混ぜ、g2(B領域ベース) と衝突させる
            F[1] = self.compose(F[1], self.get_power(self.rho_B, random.randint(1, 5)))
            G[2] = self.compose(G[2], self._generate_random_cycle(range(self.mid, self.P)))

            # 3. Girthチェック (v8.ipynb の不動点チェックを活用)
            def get_block(i, j):
                if j < self.L_half: return F[(j - i) % self.L_half]
                else: return G[(j - self.L_half - i) % self.L_half]

            is_clean = True
            # C4 チェックのみを先行させて高速化
            for i in range(self.J):
                for ip in range(i + 1, self.J):
                    for j in range(self.L_half * 2):
                        for jp in range(j + 1, self.L_half * 2):
                            if self._has_fixed_point(get_block(i, j), get_block(ip, j), 
                                                     get_block(ip, jp), get_block(i, jp)):
                                is_clean = False; break
                        if not is_clean: break
                    if not is_clean: break
            
            if is_clean:
                print(f"成功！ 試行回数: {trial}, 時間: {time.time()-start_time:.2f}s")
                return F, G

    def _has_fixed_point(self, p1, p2, p3, p4):
        # v8.ipynb 実装の高速不動点チェック
        inv_p2 = [0]*self.P; inv_p4 = [0]*self.P
        for i, v in enumerate(p2): inv_p2[v] = i
        for i, v in enumerate(p4): inv_p4[v] = i
        for x in range(self.P):
            if inv_p4[p3[inv_p2[p1[x]]]] == x: return True
        return False

# 実行
opt = ImprovedHighEntropyAPM(P=768, J=3)
F_final, G_final = opt.solve()

--- 改善版 HighEntropy 探索開始 (P=768, Girth向上モデル) ---


KeyboardInterrupt: 

In [ ]:

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    cols = np.array(p)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, cols)), shape=(size, size))

def build_matrices_internal(F, G, P, J):
    L_h = len(F)
    F_m = [tuple_to_sparse(f, P) for f in F]
    G_m = [tuple_to_sparse(g, P) for g in G]
    hx_rows = []
    hz_rows = []
    for i in range(J):
        row_x = [F_m[(j-i)%L_h] for j in range(L_h)] + [G_m[(j-i)%L_h] for j in range(L_h)]
        row_z = [G_m[(i-j)%L_h].transpose() for j in range(L_h)] + [F_m[(i-j)%L_h].transpose() for j in range(L_h)]
        hx_rows.append(hstack(row_x))
        hz_rows.append(hstack(row_z))
    return vstack(hx_rows), vstack(hz_rows)

def count_cycles_direct(H):
    num_checks, num_vars = H.shape
    adj_c = [H.getrow(i).indices for i in range(num_checks)]
    H_csc = H.tocsc()
    adj_v = [H_csc.getcol(j).indices for j in range(num_vars)]
    c4, c6 = 0, 0
    # 4-cycle
    shared = {}
    for c1 in range(num_checks):
        for v in adj_c[c1]:
            for c2 in adj_v[v]:
                if c2 > c1:
                    shared[(c1, c2)] = shared.get((c1, c2), 0) + 1
    for count in shared.values():
        if count >= 2: c4 += count * (count - 1) // 2
    # 6-cycle
    for c1 in range(num_checks):
        for v1 in adj_c[c1]:
            for c2 in adj_v[v1]:
                if c2 <= c1: continue
                for v2 in adj_c[c2]:
                    if v2 == v1: continue
                    for c3 in adj_v[v2]:
                        if c3 <= c1 or c3 == c2: continue
                        common = set(adj_c[c3]) & set(adj_c[c1])
                        for v3 in common:
                            if v3 != v1 and v3 != v2: c6 += 1
    return c4, c6 // 2

In [ ]:

# 最終確認
Hx, Hz = build_matrices_internal(F_final, G_final, P, J)
print(f"\nFinal Check - Hx Shape: {Hx.shape}")
# 直交性確認
ortho = (Hx @ Hz.transpose())
ortho.data %= 2
ortho.eliminate_zeros()
print(f"CSS Condition Violation: {ortho.nnz}")

c4, c6 = count_cycles_direct(Hx)
print(f"C4: {c4}, C6: {c6}")


Final Check - Hx Shape: (2304, 9216)
CSS Condition Violation: 72
C4: 0, C6: 9204
